# 03 · Décision et ROI
**Axes 6 et 7 · de l'erreur de prévision à la valeur économique**

Entrée : `data/processed/resultats_test.csv` et `metriques_test.csv`, produits par `02_modelisation.ipynb`. Toutes les valeurs en euros reposent sur des **hypothèses de prix posées par l'équipe**, à confirmer avec des prix de marché.

## 0 · Imports

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PROCESSED = Path("data/processed")
FIGURES = Path("reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (8, 4)

Les fichiers de résultats viennent du notebook 02. Sur Colab, si tu ouvres ce notebook dans une nouvelle session, la cellule suivante te demande de les importer.

In [ ]:
# Fichiers produits par 02_modelisation. Sur Google Colab (nouvelle session), on les importe s'ils sont absents.
fichiers = ["resultats_test.csv", "metriques_test.csv"]
manquants = [nom for nom in fichiers if not (DATA_PROCESSED / nom).exists()]

if manquants:
    try:
        from google.colab import files
        DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
        print("Importe ces fichiers :", manquants)
        uploaded = files.upload()
        for nom in uploaded:
            Path(nom).replace(DATA_PROCESSED / nom)
    except ImportError:
        raise FileNotFoundError("Lancer d'abord 02_modelisation.ipynb : il crée " + ", ".join(manquants))

## 1 · Résultats du modèle (test)

In [ ]:
# Résultats produits par 02_modelisation.ipynb (période de test : jamais vue pendant l'apprentissage)
resultats = pd.read_csv(DATA_PROCESSED / "resultats_test.csv", parse_dates=["Date"])
metriques = pd.read_csv(DATA_PROCESSED / "metriques_test.csv").set_index("indicateur")["valeur"]

mae_veille = float(metriques["mae_veille"])
mae_moyenne7 = float(metriques["mae_baseline"])
mae_modele = float(metriques["mae_modele"])

print("Période de test :", resultats["Date"].min().date(), "->", resultats["Date"].max().date(), "(", len(resultats), "jours )")
print("MAE veille :", round(mae_veille, 2), "| moyenne 7 jours :", round(mae_moyenne7, 2), "| modèle :", round(mae_modele, 2))

## 2 · Hypothèses
Acheter trop ou pas assez ne coûte pas la même chose : on rachète l'énergie manquante en urgence à 0,15 €/kWh au lieu de 0,10 €, et on revend le surplus à 0,07 €/kWh.

In [ ]:
# --- Hypothèses de l'équipe (à confirmer avec des prix de marché) ---
PRIX_ACHAT = 0.10      # €/kWh, prix prévu la veille
PRIX_URGENCE = 0.15    # €/kWh, rachat en urgence si on a sous-estimé
PRIX_REVENTE = 0.07    # €/kWh, revente du surplus si on a surestimé

PERTE_SOUS_ESTIMATION = PRIX_URGENCE - PRIX_ACHAT   # 0,05 €/kWh manquant
PERTE_SUR_ESTIMATION = PRIX_ACHAT - PRIX_REVENTE    # 0,03 €/kWh en trop

NB_FOYERS = 10000
COUT_DEVELOPPEMENT = 40000   # € la 1re année
COUT_EXPLOITATION = 10000    # € la 1re année (hébergement, suivi, réentraînement)
COUT_PROJET = COUT_DEVELOPPEMENT + COUT_EXPLOITATION

print("Perte par kWh sous-estimé :", round(PERTE_SOUS_ESTIMATION, 2), "€")
print("Perte par kWh surestimé   :", round(PERTE_SUR_ESTIMATION, 2), "€")
print("Coût du projet (1re année) :", COUT_PROJET, "€")

## 3 · Coût d'erreur par foyer et par an
Le coût est calculé jour par jour sur les prédictions du test, puis ramené à une année de 365 jours.

In [ ]:
def cout_annuel_par_foyer(reel, prevu):
    # Erreur > 0 : on a sous-estimé (rachat en urgence). Erreur < 0 : on a surestimé (revente à perte).
    erreur = reel - prevu
    cout_jour = np.where(erreur > 0, erreur * PERTE_SOUS_ESTIMATION, -erreur * PERTE_SUR_ESTIMATION)
    return cout_jour.mean() * 365


cout_veille = cout_annuel_par_foyer(resultats["reel"], resultats["pred_veille"])
cout_moyenne7 = cout_annuel_par_foyer(resultats["reel"], resultats["pred_baseline"])
cout_modele = cout_annuel_par_foyer(resultats["reel"], resultats["pred_modele"])

part_sous_modele = (resultats["reel"] > resultats["pred_modele"]).mean() * 100
part_sous_veille = (resultats["reel"] > resultats["pred_veille"]).mean() * 100

gain_foyer = round(cout_veille - cout_modele, 3)               # face à la situation actuelle
gain_foyer_moyenne7 = round(cout_moyenne7 - cout_modele, 3)    # face à la moyenne 7 jours
amelioration_mae = (mae_veille - mae_modele) / mae_veille

print("Coût d'erreur par foyer et par an :")
print("  demain = aujourd'hui :", round(cout_veille, 2), "€")
print("  moyenne 7 jours      :", round(cout_moyenne7, 2), "€")
print("  modèle               :", round(cout_modele, 2), "€")
print("Gain par foyer et par an face à la veille        :", gain_foyer, "€")
print("Gain par foyer et par an face à la moyenne 7 j   :", gain_foyer_moyenne7, "€")
print("Sous-estimation : modèle", round(part_sous_modele), "% des jours, veille", round(part_sous_veille), "% des jours")
print("Amélioration de la MAE face à la veille :", round(amelioration_mae * 100, 1), "%")

## 4 · ROI et scénarios
Formules : gain annuel = foyers × gain par foyer ; ROI (%) = (gain annuel − coût du projet) / coût du projet × 100 ; retour (mois) = coût du projet / (gain annuel / 12).

In [ ]:
def bilan(gain_par_foyer, nb_foyers=NB_FOYERS, cout_projet=COUT_PROJET):
    gain_annuel = gain_par_foyer * nb_foyers
    roi = (gain_annuel - cout_projet) / cout_projet * 100
    retour_mois = cout_projet / (gain_annuel / 12)
    return gain_annuel, roi, retour_mois


# Scénarios (axe 7)
scenarios = [
    ("Pessimiste (amélioration et écarts de prix divisés par 2)", gain_foyer * 0.5 * 0.5),
    ("Central (résultat mesuré sur le test)", gain_foyer),
    ("Optimiste (objectif de -20 % atteint)", gain_foyer * 0.20 / amelioration_mae),
    ("Référence = moyenne 7 jours", gain_foyer_moyenne7),
]

lignes = []
for nom, gain_par_foyer in scenarios:
    gain_annuel, roi, retour = bilan(gain_par_foyer)
    lignes.append({"Scénario": nom, "Gain annuel (€)": round(gain_annuel, -2),
                   "ROI an 1 (%)": round(roi, 1), "Retour (mois)": round(retour, 1)})

tableau_scenarios = pd.DataFrame(lignes)
display(tableau_scenarios)

print("Foyers nécessaires pour l'équilibre (face à la veille) :", round(COUT_PROJET / gain_foyer))
print("Foyers nécessaires pour l'équilibre (face à la moyenne 7 jours) :", round(COUT_PROJET / gain_foyer_moyenne7))

**Lecture** : face à la méthode actuelle, le projet est rentable dans les scénarios central et optimiste mais pas dans le pessimiste. Face à la moyenne des 7 derniers jours, que le modèle ne bat pas, le gain est faible et le ROI négatif : le résultat dépend fortement de la référence choisie.

## 5 · Sensibilité

In [ ]:
# Sensibilité : on fait varier une hypothèse à la fois à partir du scénario central
cas = [
    ("Scénario central", gain_foyer, NB_FOYERS, COUT_PROJET),
    ("Écarts de prix divisés par 2", gain_foyer / 2, NB_FOYERS, COUT_PROJET),
    ("Portefeuille de 5 000 foyers", gain_foyer, 5000, COUT_PROJET),
    ("Portefeuille de 1 000 foyers", gain_foyer, 1000, COUT_PROJET),
    ("Coût du projet doublé (100 000 €)", gain_foyer, NB_FOYERS, COUT_PROJET * 2),
]

lignes = []
for nom, gain_par_foyer, nb_foyers, cout_projet in cas:
    gain_annuel, roi, retour = bilan(gain_par_foyer, nb_foyers, cout_projet)
    lignes.append({"Cas": nom, "Gain annuel (€)": round(gain_annuel, -2),
                   "ROI an 1 (%)": round(roi, 1), "Retour (mois)": round(retour, 1)})

display(pd.DataFrame(lignes))

## 6 · Graphiques pour le rapport et l'oral

In [ ]:
# Graphique 1 : gain annuel par scénario
noms = ["Pessimiste", "Central", "Optimiste", "Réf. moyenne 7 j"]
gains = [gain_foyer * 0.25 * NB_FOYERS / 1000, gain_foyer * NB_FOYERS / 1000,
         gain_foyer * 0.20 / amelioration_mae * NB_FOYERS / 1000, gain_foyer_moyenne7 * NB_FOYERS / 1000]

plt.figure()
plt.bar(noms, gains)
plt.axhline(COUT_PROJET / 1000, color="orange", linestyle="--", label="Coût du projet : 50 k€")
plt.title("Gain annuel estimé par scénario (10 000 foyers)")
plt.ylabel("k€ par an")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES / "08_gain_par_scenario.png", dpi=150, bbox_inches="tight")
plt.show()

# Graphique 2 : gain net de la 1re année selon la taille du portefeuille
tailles = [1000, 2000, 5000, 10000]
gains_nets = []
for n in tailles:
    gains_nets.append((gain_foyer * n - COUT_PROJET) / 1000)

plt.figure()
plt.bar([str(n) + " foyers" for n in tailles], gains_nets)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Gain net de la 1re année selon la taille du portefeuille")
plt.ylabel("k€")
plt.tight_layout()
plt.savefig(FIGURES / "09_gain_net_portefeuille.png", dpi=150, bbox_inches="tight")
plt.show()

## 7 · Risques et recommandation
Le registre complet des risques (techniques, métier, éthiques, environnementaux), avec probabilité, impact et mesure de réduction, figure à l'axe 9 du rapport.

**Recommandation** : tester le modèle pendant 3 mois sur un petit portefeuille de foyers, avec des variables enrichies (météo, jours fériés), pour mesurer le vrai coût d'un kWh d'erreur et la vraie amélioration face aux deux références (veille et moyenne 7 jours) avant tout déploiement à grande échelle.